## 05d — Single-Bidder Inference

Scores every active Contract Notice in the silver layer with the trained XGBoost classifier and writes results to the gold layer for dashboard consumption.

**Input:** `workspace.silver.contract_notices`  
**Output:** `workspace.gold.cn_predictions`  
**Model:** `/Volumes/workspace/default/lakehouse/ml/bidders_xgb_final.pkl`

The model uses four categorical features: buyer country, CPV division, procedure type, and buyer type. Nulls in any of these columns are imputed as `UNKNOWN` before scoring. Unseen categories at inference time are handled silently via `handle_unknown='ignore'` in the OHE step. All 41,159 active CNs are scored; probability close to 0 indicates high competition.

In [0]:
%pip install xgboost

In [0]:
# Goal: load the saved model and confirm it expects the four categorical features only.
import joblib

MODEL_PATH   = "/Volumes/workspace/default/lakehouse/ml/bidders_xgb_final.pkl"
SOURCE_TABLE = "workspace.silver.contract_notices"
OUTPUT_TABLE = "workspace.gold.cn_predictions"

CAT_FEATURES = ["country_clean", "cpv_division_clean", "procedure_clean", "buyer_type_clean"]

model = joblib.load(MODEL_PATH)
print("Model loaded:", type(model.named_steps["clf"]))
print("Expected features:", CAT_FEATURES)

In [0]:
# Conclusion from previous: model loaded successfully.
# Goal: read contract notices and inspect coverage of the four feature columns
# before applying any transformations.
import pyspark.sql.functions as F

df = spark.table(SOURCE_TABLE)
print("Total CNs:", df.count())

df.select(
    F.count(F.when(F.col("buyer_country").isNull(), 1)).alias("null_buyer_country"),
    F.count(F.when(F.col("cpv_code").isNull(), 1)).alias("null_cpv_code"),
    F.count(F.when(F.col("procedure_type").isNull(), 1)).alias("null_procedure_type"),
    F.count(F.when(F.col("buyer_type").isNull(), 1)).alias("null_buyer_type"),
).show()

In [0]:
# Conclusion from previous: null counts confirm where UNKNOWN imputation will fire.
# Goal: derive the four clean features from the raw columns.
# cpv_division is the first two digits of cpv_code.
# Nulls in country, procedure_type, and buyer_type map to UNKNOWN.
# The pipeline OHE uses handle_unknown='ignore', so any category not seen during training is silently encoded as all-zeros — no additional grouping needed here.

df_features = (
    df
    .withColumn(
        "country_clean",
        F.when(F.col("buyer_country").isNull(), "UNKNOWN").otherwise(F.col("buyer_country"))
    )
    .withColumn(
        "cpv_division_clean",
        F.when(F.col("cpv_code").isNull(), "UNKNOWN")
         .otherwise(F.col("cpv_code").substr(1, 2))
    )
    .withColumn(
        "procedure_clean",
        F.when(F.col("procedure_type").isNull(), "UNKNOWN").otherwise(F.col("procedure_type"))
    )
    .withColumn(
        "buyer_type_clean",
        F.when(F.col("buyer_type").isNull(), "UNKNOWN").otherwise(F.col("buyer_type"))
    )
)

print("Feature sample:")
df_features.select("notice_id", *CAT_FEATURES).show(5, truncate=False)


In [0]:
# Conclusion from previous: four clean features derived, nulls handled.
# Goal: convert to pandas, score with the model, and attach probabilities back.

import pandas as pd
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

# bring only what is needed into pandas
cols_to_score = ["notice_id", "buyer_name", "project_title", "cpv_code",
                 "buyer_country", "estimated_value"] + CAT_FEATURES

scoring_pd = df_features.select(cols_to_score).toPandas()

X = scoring_pd[CAT_FEATURES]
scoring_pd["single_bidder_prob"] = model.predict_proba(X)[:, 1]

print(f"Scored {len(scoring_pd)} notices.")
print(scoring_pd["single_bidder_prob"].describe().round(3))

In [0]:

# Conclusion from previous: predictions computed for all CNs.
# Goal: write results to the gold output table.
# single_bidder_prob close to 0 = competitive contract (higher priority for sales).

from pyspark.sql.types import DoubleType
from datetime import datetime, timezone

output_pd = scoring_pd[[
    "notice_id", "buyer_name", "project_title", "cpv_code",
    "buyer_country", "estimated_value", "single_bidder_prob"
]].copy()

output_pd["scored_at"] = datetime.now(timezone.utc)

output_df = spark.createDataFrame(output_pd)

(output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE))

print(f"Wrote {output_df.count()} rows to {OUTPUT_TABLE}.")
spark.table(OUTPUT_TABLE).select(
    "notice_id", "buyer_name", "single_bidder_prob", "scored_at"
).show(10, truncate=False)